In [3]:
import numpy as np 
import matplotlib.pyplot as plt 
import healpy as hp
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import astropy.units as u
import rfitools
import numba as nb
import importlib
importlib.reload(rfitools)
import time

In [4]:
coords = {
        0: [79.417161473, -90.767238685, 187.9577], #MARS1
        1: [79.417198047, -90.758739192, 183.0684], #MARS2
        2: [79.388456412, -91.019202963, 25.1938], #CSA
        3: [79.418302573, -90.667395452, 59.6242], #Mars 5
        4: [79.397984238, -90.799842408, 41.6994], #Mars 6
        5: [79.411474117, -90.695266129, 31.6314], #Mars 7
        6: [79.443757694, -90.718202634, 414.9131] #MARS8
        # 7: [79.41540905, -90.77299123, 180.4816], #MARS3
        # 8: [79+24/60+28.80/3600, -90 - 48/60 - 5.09/3600, 200]
    }
antmap = {0:"MARS1", 1:"MARS2",2:"MARS4",3:"MARS5",4:"MARS6",5:"MARS7",6:"MARS8"}
ant0=EarthLocation.from_geodetic(lat=coords[0][0], lon=coords[0][1], height=coords[0][2])

In [13]:
deltat = 0.5 #sec
deltaf = 4768.37158203 #Hz
fstart = 21972656.25 # Hz
freqs = np.arange(100) * deltaf + fstart #Hz
print('Frequencies shape', freqs.shape)
print(f'From {freqs[0]} to {freqs[-1]}')
times = 1753200150 + np.arange(10)*deltat
print('Times shape', times.shape)
print(f'From {times[0]} to {times[-1]}')

Frequencies shape (100,)
From 21972656.25 to 22444725.03662097
Times shape (10,)
From 1753200150.0 to 1753200154.5


In [12]:
#==SIMULATED DATA
obstime=Time(np.arange(1753200150, 1753200150+10),format="unix",scale="utc")
ras = ['23h23m27.94s', '19h 59m 28.3566s']
decs = ['58d48m42.4s', "40° 44′ 02.096″"]
sources = SkyCoord(ra=ras,dec=decs , frame='icrs')
source_azalt = np.zeros((len(sources), 2, len(obstime)), dtype='float64')
for i,src in enumerate(sources):
    azalt = src.transform_to(AltAz(location=ant0,obstime=obstime))
    source_azalt[i, 0, :] = azalt.az.rad
    source_azalt[i, 1, :] = azalt.alt.rad
bl_enus = rfitools.get_all_bls(coords,np.arange(len(coords)))
delays0 = rfitools.geo_delay_from_enu(bl_enus, source_azalt[0,0], source_azalt[0,1])
delays1 = rfitools.geo_delay_from_enu(bl_enus, source_azalt[1,0], source_azalt[1,1])

vis = 66 * np.exp(-2j*np.pi*delays0.T[:,None, :] * freqs[None, :, None]) +\
      37 * np.exp(-2j*np.pi*delays1.T[:,None, ] * freqs[None, :, None])
print("vis shape", vis.shape)

(21, 3)
vis shape (10, 100, 21)


In [15]:
np.savez(
    '/scratch/thomasb/mapmaking_dumps/sim_vis.npz',
    vis = vis,
    mask = np.zeros_like(vis, dtype=bool),
    freqs = freqs,
    times = times
    )

In [ ]:
NSIDE=256
NPIX=hp.nside2npix(NSIDE)
print("resol", hp.nside2resol(NSIDE,arcmin=True), "npix", NPIX)
co_dec,ra=hp.pix2ang(NSIDE,np.arange(NPIX))
dec=np.pi/2-co_dec

src = SkyCoord(ra=ra*u.rad, dec=dec*u.rad, frame='icrs')
altaz_hp =src.transform_to(AltAz(location=ant0,obstime=obstime[0]))
delays = rfitools.geo_delay_from_enu(bl_enus, altaz_hp.az.rad, altaz_hp.alt.rad)

In [ ]:
delaysT = delays[:,:].T.copy() #exclude MARS1-2
data = vis[0, :, :].T.copy()

In [ ]:
@nb.njit(parallel=True)
def get_map(data,freqs,delays,npix):
    print(data.shape, freqs.shape, delays.shape)
    map1 = np.zeros(npix, dtype=np.float64)
    nfreq, nbl = data.shape
    N_vis = nfreq * nbl
    for p in nb.prange(npix):
        pixel_sum = 0.
        for f in range(nfreq):
            nu = freqs[f]
            for b in range(nbl):
                tau = delays[p, b] #delay shape is npix, nbl for CPU

                # Calculate the fringe factor for this specific visibility
                fringe = np.exp(2j * np.pi * nu * tau)
                
                # Accumulate the dot product
                pixel_sum += fringe.real * data[f, b].real - fringe.imag * data[f, b].imag
        
        # Calculate the mean and assign to the pixel map
        map1[p] = pixel_sum / N_vis
    return map1

In [ ]:
t1=time.time()
map1 = get_map(data,freqs,delaysT,NPIX)
t2=time.time()
print(t2-t1)

In [ ]:
hp.orthview(map1,min=2,max=6,rot=(0,90),half_sky=True)
hp.graticule(dpar=15,  dmer=15, color='white', alpha=0.5, ls=':')